# 🚁 Drone Sound Detection Using Audio Classification

**A Beginner-Friendly Machine Learning Project**

> **Goal:** Build a system that listens to audio and decides whether a drone is present.

---

## Table of Contents
1. [Problem Statement](#1-problem-statement)
2. [Dataset Overview](#2-dataset-overview)
3. [Setup & Imports](#3-setup--imports)
4. [Load Audio Data](#4-load-audio-data)
5. [Visualise Audio](#5-visualise-audio)
6. [Extract Features](#6-extract-features)
7. [Split Dataset](#7-split-dataset)
8. [Model 1 – Random Forest](#8-model-1--random-forest)
9. [Model 2 – CNN on Mel Spectrograms](#9-model-2--cnn-on-mel-spectrograms)
10. [Model Comparison](#10-model-comparison)
11. [Future Improvements](#11-future-improvements)
12. [Conclusion](#12-conclusion)

## 1. Problem Statement

Drones (Unmanned Aerial Vehicles) are being used for photography, delivery, surveillance, and even warfare. Detecting an approaching drone using only sound is a **non-invasive** and **cost-effective** method for security systems.

| Aspect | Details |
|--------|---------|
| **Task** | Binary audio classification |
| **Class 0** | `non_drone` – ambient sounds (wind, traffic, birds) |
| **Class 1** | `drone` – UAV rotor / propeller noise |
| **Input** | Short audio clip (.wav) |
| **Output** | Probability that a drone is present |

We will train **two models** and compare them:
- 🌲 **Random Forest** – classical ML on handcrafted features
- 🧠 **CNN** – deep learning on mel spectrogram images

## 2. Dataset Overview

We use the **DADS (Drone Audio Detection Samples)** dataset.

**Download instructions:**
```bash
# Using Kaggle CLI (recommended)
kaggle datasets download -d sherildavid/drone-audio-detection-samples-dads
unzip drone-audio-detection-samples-dads.zip -d ../data/
```

Place files in:
- `data/drone/` — audio clips containing drone sounds
- `data/non_drone/` — audio clips without drone sounds

Each clip is a `.wav` file, typically 1–5 seconds long, recorded in various environments.

## 3. Setup & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# ── Numerical computing ───────────────────────────────────────
import numpy as np
import pandas as pd

# ── Audio processing ──────────────────────────────────────────
import librosa
import librosa.display

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# ── Machine learning ─────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_score,
                              recall_score, f1_score)
import joblib

# ── Deep learning ─────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# ── Progress bar ─────────────────────────────────────────────
from tqdm.notebook import tqdm

# ── Add src/ to path so we can import our modules ────────────
sys.path.insert(0, os.path.join('..', 'src'))

# ── Reproducibility ──────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ── Constants ────────────────────────────────────────────────
SAMPLE_RATE  = 22050   # Hz
DURATION     = 3.0     # seconds
N_MFCC       = 40
N_MELS       = 128
HOP_LENGTH   = 512
N_FFT        = 2048

DRONE_DIR     = os.path.join('..', 'data', 'drone')
NON_DRONE_DIR = os.path.join('..', 'data', 'non_drone')

print('✅ All libraries imported successfully!')
print(f'TensorFlow version : {tf.__version__}')
print(f'Librosa version    : {librosa.__version__}')

## 4. Load Audio Data

We will:
1. Scan `data/drone/` and `data/non_drone/` for `.wav` files
2. Load each file with Librosa
3. **Resample** to 22,050 Hz for consistency
4. **Pad or trim** to exactly 3 seconds

In [ ]:
def collect_files(drone_dir, non_drone_dir):
    """Scan folders and return a labelled DataFrame."""
    records = []
    for folder, cls in [(drone_dir, 'drone'), (non_drone_dir, 'non_drone')]:
        if not os.path.isdir(folder):
            print(f'⚠️  Missing: {folder}')
            continue
        for fname in os.listdir(folder):
            if fname.lower().endswith(('.wav', '.mp3', '.flac')):
                records.append({'filepath': os.path.join(folder, fname),
                                'class_name': cls,
                                'label': 1 if cls == 'drone' else 0})
    df = pd.DataFrame(records)
    return df

df = collect_files(DRONE_DIR, NON_DRONE_DIR)

print(f'Total files : {len(df)}')
print(df['class_name'].value_counts())
df.head()

In [ ]:
def load_audio(filepath, sr=SAMPLE_RATE, duration=DURATION):
    """
    Load a .wav file, resample, and pad/trim to `duration` seconds.
    Returns a numpy array of audio samples.
    """
    try:
        y, _ = librosa.load(filepath, sr=sr, duration=duration,
                            res_type='kaiser_fast')
    except Exception:
        return np.zeros(int(sr * duration), dtype=np.float32)

    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    return y


print('Loading all audio files...')
waveforms = []
labels = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    waveforms.append(load_audio(row['filepath']))
    labels.append(row['label'])

waveforms = np.array(waveforms)
labels    = np.array(labels)

print(f'Waveforms shape : {waveforms.shape}  ({waveforms.shape[0]} clips × {waveforms.shape[1]} samples)')
print(f'Labels shape    : {labels.shape}')
print(f'Drone clips     : {(labels == 1).sum()}')
print(f'Non-drone clips : {(labels == 0).sum()}')

## 5. Visualise Audio

Looking at audio in different ways helps us understand what features to extract.

| Plot | What it shows |
|------|---------------|
| **Waveform** | Amplitude over time – how loud the sound is |
| **Spectrogram** | Frequency content over time (STFT) |
| **Mel Spectrogram** | Like a spectrogram but on the perceptual mel scale |
| **MFCCs** | Compact fingerprint of the audio texture |

In [ ]:
# Helper: pick one sample from each class
drone_idx     = np.where(labels == 1)[0][0]
non_drone_idx = np.where(labels == 0)[0][0]

y_drone     = waveforms[drone_idx]
y_non_drone = waveforms[non_drone_idx]

# ── Plot 1: Waveforms ─────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 5), facecolor='#1e1e2e')
for ax, y, title, colour in zip(
        axes,
        [y_drone, y_non_drone],
        ['🚁 Drone Waveform', '🌿 Non-Drone Waveform'],
        ['#89b4fa', '#a6e3a1']):
    librosa.display.waveshow(y, sr=SAMPLE_RATE, ax=ax, color=colour)
    ax.set_facecolor('#1e1e2e')
    ax.set_title(title, color='#cdd6f4')
    ax.set_xlabel('Time (s)', color='#cdd6f4')
    ax.set_ylabel('Amplitude', color='#cdd6f4')
    ax.tick_params(colors='#cdd6f4')
    ax.spines[:].set_color('#313244')

plt.suptitle('Waveform Comparison', color='#cba6f7', fontsize=14)
plt.tight_layout()
os.makedirs(os.path.join('..', 'results', 'figures'), exist_ok=True)
plt.savefig(os.path.join('..', 'results', 'figures', 'waveforms.png'), dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Mel Spectrograms ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor='#1e1e2e')

for ax, y, title in zip(axes,
                         [y_drone, y_non_drone],
                         ['🚁 Drone Mel Spectrogram', '🌿 Non-Drone Mel Spectrogram']):
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE,
                                          n_mels=N_MELS, n_fft=N_FFT,
                                          hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=SAMPLE_RATE,
                                    hop_length=HOP_LENGTH,
                                    x_axis='time', y_axis='mel',
                                    ax=ax, cmap='inferno')
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    ax.set_facecolor('#1e1e2e')
    ax.set_title(title, color='#cdd6f4')
    ax.tick_params(colors='#cdd6f4')
    ax.spines[:].set_color('#313244')

plt.suptitle('Mel Spectrogram Comparison', color='#cba6f7', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'mel_spectrograms.png'), dpi=150)
plt.show()

In [ ]:
# ── Plot 3: MFCCs ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor='#1e1e2e')

for ax, y, title in zip(axes,
                         [y_drone, y_non_drone],
                         ['🚁 Drone MFCCs', '🌿 Non-Drone MFCCs']):
    mfccs = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                   n_fft=N_FFT, hop_length=HOP_LENGTH)
    img = librosa.display.specshow(mfccs, sr=SAMPLE_RATE,
                                    hop_length=HOP_LENGTH,
                                    x_axis='time', ax=ax, cmap='coolwarm')
    fig.colorbar(img, ax=ax)
    ax.set_facecolor('#1e1e2e')
    ax.set_title(title, color='#cdd6f4')
    ax.set_ylabel('MFCC Coefficient', color='#cdd6f4')
    ax.tick_params(colors='#cdd6f4')

plt.suptitle('MFCC Comparison (40 coefficients)', color='#cba6f7', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'mfccs.png'), dpi=150)
plt.show()

print('💡 Notice how the MFCC pattern differs between drone and non-drone!')
print('   The drone pattern is more regular (periodic rotor vibration).')

## 6. Extract Features

For **classical ML** (Random Forest, SVM) we need fixed-size feature vectors.

We extract per clip:
| Feature | Dimension | Description |
|---------|-----------|-------------|
| MFCC mean | 40 | Average shape of the spectrum |
| MFCC std  | 40 | Variability of the spectrum |
| Chroma mean | 12 | Musical pitch class energies |
| Spectral Centroid | 1 | "Brightness" of the sound |
| Zero-Crossing Rate | 1 | Signal noisiness |
| **Total** | **94** | — |

In [ ]:
def extract_features(y, sr=SAMPLE_RATE):
    """Extract a 94-dimensional feature vector from one audio clip."""
    feats = []

    # 1. MFCCs – mean and std over time (80 values)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                  n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats.append(np.mean(mfccs, axis=1))  # (40,)
    feats.append(np.std(mfccs,  axis=1))  # (40,)

    # 2. Chroma – mean (12 values)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr,
                                          n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats.append(np.mean(chroma, axis=1))  # (12,)

    # 3. Spectral Centroid – mean (1 value)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr,
                                                  n_fft=N_FFT,
                                                  hop_length=HOP_LENGTH)
    feats.append(np.mean(centroid))  # scalar

    # 4. Zero-Crossing Rate – mean (1 value)
    zcr = librosa.feature.zero_crossing_rate(y=y, hop_length=HOP_LENGTH)
    feats.append(np.mean(zcr))  # scalar

    return np.concatenate([np.atleast_1d(f) for f in feats])  # (94,)


# Build feature matrix for all clips
print('Extracting features...')
X_feat = np.array([extract_features(y) for y in tqdm(waveforms)])

print(f'\nFeature matrix shape : {X_feat.shape}')
print(f'Each clip → {X_feat.shape[1]}-dimensional feature vector')

In [ ]:
# Build mel spectrogram array for CNN
def make_mel_spec(y, sr=SAMPLE_RATE, n_mels=128, fixed_len=128):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
                                          n_fft=N_FFT, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    if mel_db.shape[1] < fixed_len:
        mel_db = np.pad(mel_db, ((0,0),(0, fixed_len-mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :fixed_len]
    return mel_db  # (128, 128)


print('Computing mel spectrograms for CNN...')
X_spec = np.array([make_mel_spec(y) for y in tqdm(waveforms)])
X_spec = X_spec[..., np.newaxis]  # add channel dim → (N, 128, 128, 1)

# Normalise to [0, 1]
X_spec = (X_spec - X_spec.min()) / (X_spec.max() - X_spec.min() + 1e-8)

print(f'Spectrogram array shape : {X_spec.shape}')

## 7. Split Dataset

We split **80% train / 20% test** for classical ML, and **70% / 15% / 15%** (train/val/test) for CNN.

- `stratify=labels` ensures both classes are equally represented in each split
- `random_state=42` makes the split reproducible

In [ ]:
# Classical ML split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_feat, labels, test_size=0.2, random_state=RANDOM_SEED, stratify=labels)

print('Classical ML split')
print(f'  Train : {len(X_tr)} clips')
print(f'  Test  : {len(X_te)} clips')

# CNN split
X_tmp, X_cnn_te, y_tmp, y_cnn_te = train_test_split(
    X_spec, labels, test_size=0.15, random_state=RANDOM_SEED, stratify=labels)
X_cnn_tr, X_cnn_val, y_cnn_tr, y_cnn_val = train_test_split(
    X_tmp, y_tmp, test_size=0.176, random_state=RANDOM_SEED, stratify=y_tmp)

print('\nCNN split')
print(f'  Train : {len(X_cnn_tr)} clips')
print(f'  Val   : {len(X_cnn_val)} clips')
print(f'  Test  : {len(X_cnn_te)} clips')

# Scale features for classical ML
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)
print('\n✅ Data split and scaled.')

## 8. Model 1 – Random Forest

Random Forest is an **ensemble** of decision trees. Each tree votes on the class, and the majority wins.

**Pros:** fast, interpretable, works well with small datasets  
**Cons:** can't learn spatial patterns like a CNN

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=None,
                             min_samples_leaf=2, class_weight='balanced',
                             n_jobs=-1, random_state=RANDOM_SEED)
rf.fit(X_tr_s, y_tr)
print('✅ Random Forest trained!')

# 5-fold cross-validation
cv = cross_val_score(rf, X_tr_s, y_tr, cv=5, scoring='f1')
print(f'   5-fold CV F1: {cv.mean():.3f} ± {cv.std():.3f}')

In [ ]:
# Evaluate RF
y_pred_rf = rf.predict(X_te_s)

print('Random Forest – Test Set Evaluation')
print('=' * 45)
print(classification_report(y_te, y_pred_rf, target_names=['non_drone', 'drone']))

cm_rf = confusion_matrix(y_te, y_pred_rf)

fig, ax = plt.subplots(figsize=(5, 4), facecolor='#1e1e2e')
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Drone', 'Drone'],
            yticklabels=['Non-Drone', 'Drone'], ax=ax)
ax.set_facecolor('#1e1e2e')
ax.set_title('Confusion Matrix – Random Forest', color='#cdd6f4')
ax.set_xlabel('Predicted', color='#cdd6f4')
ax.set_ylabel('True', color='#cdd6f4')
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'cm_rf.png'), dpi=150)
plt.show()

rf_metrics = {
    'accuracy' : accuracy_score(y_te, y_pred_rf),
    'precision': precision_score(y_te, y_pred_rf),
    'recall'   : recall_score(y_te, y_pred_rf),
    'f1'       : f1_score(y_te, y_pred_rf)
}
print(rf_metrics)

In [ ]:
# Feature importance – which of the 94 features matter most?
importances = rf.feature_importances_
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]

feature_names = (
    [f'MFCC_mean_{i}' for i in range(N_MFCC)] +
    [f'MFCC_std_{i}'  for i in range(N_MFCC)] +
    [f'Chroma_{i}'    for i in range(12)] +
    ['SpectralCentroid', 'ZCR']
)

fig, ax = plt.subplots(figsize=(10, 5), facecolor='#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.barh([feature_names[i] for i in top_idx[::-1]],
         importances[top_idx[::-1]],
         color='#89b4fa', alpha=0.85)
ax.set_title(f'Top {top_n} Feature Importances (Random Forest)', color='#cba6f7')
ax.set_xlabel('Importance', color='#cdd6f4')
ax.tick_params(colors='#cdd6f4')
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'feature_importance.png'), dpi=150)
plt.show()

## 9. Model 2 – CNN on Mel Spectrograms

The CNN treats each mel spectrogram as a **128×128 image** and learns visual patterns automatically.

**Architecture:**
```
Input (128×128×1)
  → Conv2D(32) → BatchNorm → MaxPool → Dropout
  → Conv2D(64) → BatchNorm → MaxPool → Dropout
  → Conv2D(128) → BatchNorm → GlobalAvgPool
  → Dense(128) → Dropout
  → Dense(1) → Sigmoid → P(drone)
```

In [ ]:
def build_cnn(input_shape=(128, 128, 1)):
    model = keras.Sequential([
        keras.layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),

        # Classification head
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ], name='DroneCNN')
    return model


cnn = build_cnn()
cnn.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)
cnn.summary()

In [ ]:
os.makedirs(os.path.join('..', 'models'), exist_ok=True)
cnn_path = os.path.join('..', 'models', 'cnn_model.keras')

cbs = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(cnn_path, monitor='val_loss',
                               save_best_only=True, verbose=0),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                 patience=5, min_lr=1e-6, verbose=0)
]

print('Training CNN...')
history = cnn.fit(
    X_cnn_tr, y_cnn_tr,
    validation_data=(X_cnn_val, y_cnn_val),
    epochs=50,
    batch_size=32,
    callbacks=cbs
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor='#1e1e2e')

for ax, metric, val_metric, ylabel in zip(
        axes,
        ['loss', 'accuracy'], ['val_loss', 'val_accuracy'],
        ['Binary Cross-Entropy', 'Accuracy']):
    ax.set_facecolor('#1e1e2e')
    ax.plot(history.history[metric],     label='Train', color='#89b4fa')
    ax.plot(history.history[val_metric], label='Val',   color='#f38ba8')
    ax.set_title(metric.capitalize(), color='#cdd6f4')
    ax.set_xlabel('Epoch', color='#cdd6f4')
    ax.set_ylabel(ylabel, color='#cdd6f4')
    ax.legend()
    ax.tick_params(colors='#cdd6f4')
    ax.grid(alpha=0.2)

plt.suptitle('CNN Training History', color='#cba6f7', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'cnn_history.png'), dpi=150)
plt.show()

In [ ]:
# Evaluate CNN
y_prob_cnn = cnn.predict(X_cnn_te, verbose=0).flatten()
y_pred_cnn = (y_prob_cnn >= 0.5).astype(int)

print('CNN – Test Set Evaluation')
print('=' * 45)
print(classification_report(y_cnn_te, y_pred_cnn, target_names=['non_drone', 'drone']))

cm_cnn = confusion_matrix(y_cnn_te, y_pred_cnn)

fig, ax = plt.subplots(figsize=(5, 4), facecolor='#1e1e2e')
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Drone', 'Drone'],
            yticklabels=['Non-Drone', 'Drone'], ax=ax)
ax.set_facecolor('#1e1e2e')
ax.set_title('Confusion Matrix – CNN', color='#cdd6f4')
ax.set_xlabel('Predicted', color='#cdd6f4')
ax.set_ylabel('True', color='#cdd6f4')
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'cm_cnn.png'), dpi=150)
plt.show()

cnn_metrics = {
    'accuracy' : accuracy_score(y_cnn_te, y_pred_cnn),
    'precision': precision_score(y_cnn_te, y_pred_cnn),
    'recall'   : recall_score(y_cnn_te, y_pred_cnn),
    'f1'       : f1_score(y_cnn_te, y_pred_cnn)
}
print(cnn_metrics)

## 10. Model Comparison

Let's put both models head-to-head.

In [ ]:
# Comparison table
all_metrics = {'Random Forest': rf_metrics, 'CNN': cnn_metrics}

comp_df = pd.DataFrame(all_metrics).T
print('Model Comparison Table')
print('=' * 55)
print(comp_df.to_string())

best = comp_df['f1'].idxmax()
print(f'\n🏆 Best model by F1-score: {best}')

In [ ]:
# Grouped bar chart
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']
labels_plot     = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(labels_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5), facecolor='#1e1e2e')
ax.set_facecolor('#1e1e2e')

for i, (model_name, colour) in enumerate(
        zip(all_metrics, ['#89b4fa', '#a6e3a1'])):
    vals = [all_metrics[model_name][m] for m in metrics_to_plot]
    bars = ax.bar(x + i*width, vals, width, label=model_name,
                  color=colour, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01, f'{v:.3f}',
                ha='center', fontsize=9, color='#cdd6f4')

ax.set_xticks(x + width/2)
ax.set_xticklabels(labels_plot, color='#cdd6f4')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', color='#cdd6f4')
ax.set_title('Model Comparison', color='#cba6f7', fontsize=14)
ax.legend(facecolor='#313244', labelcolor='#cdd6f4')
ax.tick_params(colors='#cdd6f4')
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'model_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Save metrics to JSON
with open(os.path.join('..', 'results', 'metrics.json'), 'w') as f:
    json.dump(all_metrics, f, indent=4)
print('📄 Metrics saved to results/metrics.json')

# Save models
os.makedirs(os.path.join('..', 'models'), exist_ok=True)
joblib.dump(rf, os.path.join('..', 'models', 'random_forest_model.pkl'))
joblib.dump(scaler, os.path.join('..', 'models', 'random_forest_scaler.pkl'))
print('💾 Random Forest model saved to models/')
# CNN was already saved by ModelCheckpoint callback

## 11. Future Improvements

Here are 7 concrete ways to make this system better:

| # | Idea | Impact |
|---|------|--------|
| 1 | **Data augmentation** (pitch shift, noise injection, time stretch) | High – more diverse training data |
| 2 | **Real-time detection** with sliding window on microphone stream | High – practical deployment |
| 3 | **Multi-class classification** (DJI, Parrot, military UAV, etc.) | Medium – finer-grained alerts |
| 4 | **Audio Spectrogram Transformer (AST)** | High – state-of-the-art accuracy |
| 5 | **CRNN (CNN + LSTM)** for temporal modelling | Medium – better for long audio |
| 6 | **Edge deployment** (Raspberry Pi, Jetson Nano) | High – real-world use |
| 7 | **Ensemble** RF predictions + CNN predictions | Medium – improved robustness |

### Real-world deployment considerations
- **False Positives (FP):** System thinks drone is present when it isn't → alert fatigue
- **False Negatives (FN):** System misses a real drone → security risk (**more critical!**)
- For security applications, **tune the decision threshold** lower than 0.5 to reduce FN

## 12. Conclusion

### What we built
A complete end-to-end pipeline for **audio-based drone detection**:

1. ✅ Loaded and preprocessed WAV audio files with **Librosa**
2. ✅ Visualised waveforms, spectrograms, mel spectrograms, and MFCCs
3. ✅ Extracted 94-dimensional handcrafted feature vectors
4. ✅ Trained a **Random Forest** classsifier (classical ML baseline)
5. ✅ Trained a **CNN** on mel spectrogram images (deep learning)
6. ✅ Evaluated both with accuracy, precision, recall, F1, confusion matrices
7. ✅ Compared models and identified the best performer

### Key takeaways for students
- **Feature engineering matters** – even classical ML can perform well with good features
- **CNNs learn features automatically** from raw spectrogram images
- **Mel spectrograms** are the bridge between raw audio and computer vision  
- **Always check the confusion matrix** – accuracy alone can be misleading on imbalanced datasets
- **EarlyStopping + ModelCheckpoint** are essential training callbacks that save time and prevent overfitting

---

*Built with ❤️ as a student portfolio ML project*